In [0]:
print("Spark - Window Functions")

In [0]:
# 1. row_number
# 2. rank 
# 3. dense_rank
# 4. ntile
# 5. lead
# 6. lag
# 7. first_value
# 8. last_value
# 10. sum
# 11. avg
# 12. max
# 13. min
# 14. count

In [0]:
ord_df = spark.sql("select * from samples.tpch.orders")
ord_df.display()

In [0]:
ord_df = spark.table("samples.tpch.orders")
ord_df.display()

In [0]:
ord_df.createOrReplaceTempView("orders")

In [0]:
%sql
select * from orders
order by o_orderdate desc
limit 1;

In [0]:
%sql
select 
o_custkey, o_orderkey,
max(o_orderdate) as max_order_date
from orders
group by o_custkey, o_orderkey
order by max_order_date desc;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
windowSpec = Window.partitionBy("o_custkey").orderBy(F.col("o_orderdate").desc())
ran_ord_df = ord_df.withColumn("rnk", F.dense_rank().over(windowSpec))
display(ran_ord_df)

In [0]:
ran_ord_df = (
    ran_ord_df
    .filter(F.col("rnk") == 1)
    .drop("rnk")
)
ran_ord_df.display()

In [0]:
%sql
select 
*
from orders
qualify dense_rank() over (partition by o_custkey order by o_orderdate desc) = 1;

In [0]:
query = """
select 
*
from orders
qualify dense_rank() over (partition by o_custkey order by o_orderdate desc) = 1
"""

rnk_ord_df = spark.sql(query)
display(rnk_ord_df)